In [0]:
%run ./01_logging_utils.py

In [0]:
from pyspark.sql import functions as F, Window
from functools import reduce

silver = spark.table(T_SILVER)
if silver.limit(1).count() == 0:
    raise Exception("Silver is empty. Run 04_silver_transform first.")

In [0]:
STABLES = ["tether", "usd-coin", "dai", "paxos-standard", "frax"]
dc = (silver.groupBy("coin_id")
      .agg(F.min("event_ts").alias("first_seen_ts"), F.max("event_ts").alias("last_seen_ts"))
      .withColumn("coin_key", F.xxhash64("coin_id"))
      .withColumn("coin_name", F.initcap(F.regexp_replace("coin_id", "-", " ")))
      .withColumn("is_stablecoin", F.col("coin_id").isin(STABLES))
      .withColumn("load_timestamp", F.current_timestamp()))
run_gold_merge("dim_coin", dc, T_DIM_COIN, ["coin_key"],
               exact=["coin_id", "coin_name", "is_stablecoin", "first_seen_ts", "last_seen_ts"])

In [0]:
b = silver.agg(F.min("event_date").alias("mn"), F.max("event_date").alias("mx")).first()
dd = (spark.sql(f"SELECT explode(sequence(to_date('{b['mn']}'), to_date('{b['mx']}'), interval 1 day)) AS full_date")
      .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
      .withColumn("year", F.year("full_date"))
      .withColumn("quarter", F.quarter("full_date"))
      .withColumn("month", F.month("full_date"))
      .withColumn("month_name", F.date_format("full_date", "MMMM"))
      .withColumn("day_name", F.date_format("full_date", "EEEE"))
      .withColumn("week_of_year", F.weekofyear("full_date"))
      .withColumn("day_of_month", F.dayofmonth("full_date"))
      .withColumn("day_of_week", F.dayofweek("full_date"))
      .withColumn("is_weekend", F.dayofweek("full_date").isin(1, 7)))
run_gold_merge("dim_date", dd, T_DIM_DATE, ["date_key"],
               exact=["full_date", "year", "quarter", "month", "month_name", "day_name",
                      "week_of_year", "day_of_month", "day_of_week", "is_weekend"])

In [0]:
dh = (spark.range(24).select(F.col("id").cast("int").alias("hour_key"))
      .withColumn("hour_label", F.format_string("%02d:00", "hour_key"))
      .withColumn("day_part",
                  F.when(F.col("hour_key") < 6, "Night")
                   .when(F.col("hour_key") < 12, "Morning")
                   .when(F.col("hour_key") < 18, "Afternoon")
                   .otherwise("Evening")))
run_gold_merge("dim_hour", dh, T_DIM_HOUR, ["hour_key"], exact=["hour_label", "day_part"])

In [0]:
coins = spark.table(T_DIM_COIN).select("coin_key", "coin_id", "is_stablecoin")
n_all    = coins.count()
n_stable = coins.filter("is_stablecoin").count()
n_other  = n_all - n_stable
alts     = coins.filter((~F.col("is_stablecoin")) & (~F.col("coin_id").isin("bitcoin", "ethereum")))
n_alts   = alts.count()

latest = (silver.withColumn("rn", F.row_number().over(
              Window.partitionBy("coin_id").orderBy(F.col("event_ts").desc())))
          .filter("rn = 1").select("coin_id", "market_cap"))
top5 = (coins.join(latest, "coin_id").filter(F.col("market_cap").isNotNull())
        .orderBy(F.col("market_cap").desc()).limit(5))

SCEN = [
 ("Equal weight (all coins)", "Same weight in every coin"),
 ("BTC 60 / ETH 40", "60% Bitcoin, 40% Ethereum"),
 ("Top 5 by market cap", "Equal weight in the 5 coins with the highest latest market cap"),
 ("Stablecoin heavy", "50% split across stablecoins, 50% across all other coins"),
 ("Altcoin heavy", "Equal weight in non-stable coins excluding Bitcoin and Ethereum"),
]
S = [s[0] for s in SCEN]

parts = [
  coins.select(F.lit(S[0]).alias("scenario_name"), "coin_key", (F.lit(1.0) / n_all).alias("weight")),
  coins.filter(F.col("coin_id").isin("bitcoin", "ethereum"))
       .select(F.lit(S[1]).alias("scenario_name"), "coin_key",
               F.when(F.col("coin_id") == "bitcoin", 0.6).otherwise(0.4).alias("weight")),
  top5.select(F.lit(S[2]).alias("scenario_name"), "coin_key", F.lit(1.0 / 5).alias("weight")),
  coins.select(F.lit(S[3]).alias("scenario_name"), "coin_key",
               F.when(F.col("is_stablecoin"), 0.5 / max(n_stable, 1))
                .otherwise(0.5 / max(n_other, 1)).alias("weight")),
  alts.select(F.lit(S[4]).alias("scenario_name"), "coin_key",
              (F.lit(1.0) / max(n_alts, 1)).alias("weight")),
]
bridge = (reduce(lambda a, b: a.unionByName(b), parts)
          .withColumn("scenario_key", F.xxhash64("scenario_name"))
          .select("scenario_key", "coin_key", "weight"))
dsc = (spark.createDataFrame(SCEN, "scenario_name string, description string")
       .withColumn("scenario_key", F.xxhash64("scenario_name")))

run_gold_merge("dim_scenario", dsc, T_DIM_SCENARIO, ["scenario_key"],
               exact=["scenario_name", "description"], delete_unmatched=True)
run_gold_merge("bridge_scenario_coin", bridge, T_BRIDGE, ["scenario_key", "coin_key"],
               approx=["weight"], delete_unmatched=True)

In [0]:
for t in [T_DIM_COIN, T_DIM_DATE, T_DIM_HOUR, T_DIM_SCENARIO, T_BRIDGE]:
    print(t.split(".")[-1], spark.table(t).count())
display(spark.sql(f"""SELECT s.scenario_name, count(*) coins, round(sum(b.weight), 6) weight_sum
                      FROM {T_BRIDGE} b JOIN {T_DIM_SCENARIO} s USING (scenario_key)
                      GROUP BY 1 ORDER BY 1"""))
display(spark.sql(f"SELECT * FROM {T_LOGS} WHERE layer='gold' ORDER BY start_time DESC LIMIT 6"))